# DNP3 Intrusion Detection System — Feature Extractor
## Unsupervised Learning Pipeline

**Project:** DNP3 IDS using ML on real SCADA/OT network traffic  
**Dataset:** IEEE Dataport — DNP3 Intrusion Detection Dataset (UOWM)  
**Objective:** Extract packet-level features from raw PCAP files and label them by attack category for unsupervised ML modeling (KMeans, GMM, Learning Automata Clustering)

---

### Pipeline Overview

```
PCAP Files (IEEE Dataport)
       │
       ▼
PCAP Discovery  →  Label Mapping  →  Port Verification
       │
       ▼
parse_pcap_files()  →  Scapy Packet Parser
       │
       ├── TCP/IP filter
       ├── DNP3 port filter  (20000–20003)
       ├── Magic bytes check (0x05 0x64)
       └── Feature extraction (31 columns)
       │
       ▼
DataFrame  →  Save CSV  →  Google Drive
```


## Step 1 — Install Dependencies

Install `scapy` — the core library for reading and parsing raw PCAP network traffic files at the packet level.


In [1]:
!pip install scapy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 11.5 MB/s eta 0:00:00


## Step 2 — Import Libraries

Import the full data science stack. Key libraries:
- **scapy** — reads PCAP files and parses TCP/IP/DNP3 packet layers
- **tqdm** — progress bar during PCAP processing
- **tabulate** — pretty-print tabular data in console
- **pandas / numpy** — data manipulation and feature storage


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import joblib, os
from tqdm.auto import tqdm

import os
import pandas as pd
from scapy.all import rdpcap, TCP, IP, Ether, Raw

# !pip install tabulate
from tabulate import tabulate # For pretty-printing tabular data in console

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)


## Step 3 — DNP3 Protocol Definitions & Core Parser

### 3.1 DNP3 Function Code Dictionary

DNP3 application-layer function codes are integer values (0–130). This dictionary maps each integer to its human-readable name so that extracted features are interpretable rather than raw numbers.

Key function codes relevant to attacks in this dataset:

| Code | Name | Relevance |
|---|---|---|
| 1 | READ | Normal polling |
| 13 | COLD_RESTART | Attack — forces device reboot |
| 14 | WARM_RESTART | Attack — soft restart |
| 15 | INITIALIZE_DATA | Attack — wipes data |
| 18 | STOP_APPL | Attack — stops application |
| 21 | DISABLE_UNSOLICITED | Attack — disables event reporting |
| 129 | RESPONSE | Normal — outstation reply |
| 130 | UNSOLICITED_RESPONSE | Normal — event-driven reply |

### 3.2 Core PCAP Parser (`parse_pcap_files`)

This is the main extraction engine. It accepts a dictionary of `{pcap_path: label}` and returns a DataFrame of extracted features.

**Filtering pipeline per packet:**
1. Must have TCP + IP layers
2. Must be on a DNP3 port (`{20000, 20001, 20002, 20003}`)
3. Must have a Raw payload
4. Must start with DNP3 magic bytes `0x05 0x64`

**Features extracted per packet (31 columns):**

| Layer | Features |
|---|---|
| Frame | number, time_epoch, time_relative, time_delta, len, cap_len |
| Ethernet | eth.src, eth.dst |
| IP | src, dst, proto, ttl |
| TCP | srcport, dstport, flags, len, window_size, checksum, time_delta |
| DNP3 Data Link | start, len, ctrl, dst_addr, src_addr, DIR, PRM, FCB, FCV bits, func_code_link |
| DNP3 Application | func_code, func_name, payload_len, raw_hex |
| Label | attack category |


In [3]:
# ── DNP3 Function Code Names (per DNP3 spec) ──────────────────────────────────
DNP3_FUNC_NAMES = {
    0: "CONFIRM", 1: "READ", 2: "WRITE", 3: "SELECT",
    4: "OPERATE", 5: "DIRECT_OPERATE", 6: "DIRECT_OPERATE_NR",
    7: "IMMED_FREEZE", 8: "IMMED_FREEZE_NR", 9: "FREEZE_CLEAR",
    10: "FREEZE_CLEAR_NR", 11: "FREEZE_AT_TIME", 12: "FREEZE_AT_TIME_NR",
    13: "COLD_RESTART", 14: "WARM_RESTART", 15: "INITIALIZE_DATA",
    16: "INITIALIZE_APPL", 17: "START_APPL", 18: "STOP_APPL",
    19: "SAVE_CONFIG", 20: "ENABLE_UNSOLICITED", 21: "DISABLE_UNSOLICITED",
    22: "ASSIGN_CLASS", 23: "DELAY_MEASURE", 24: "RECORD_CURRENT_TIME",
    129: "RESPONSE", 130: "UNSOLICITED_RESPONSE",
}


def parse_pcap_files(pcap_label_map: dict) -> pd.DataFrame:
    """
    Parse all PCAP files in pcap_label_map and extract DNP3 packet-level features.

    Args:
        pcap_label_map: dict of {full_pcap_path: attack_label}

    Returns:
        pd.DataFrame with 31 feature columns + label
    """
    dnp3_rows   = []
    pkt_counter = 0
    # NOTE: first_time / last_time are no longer initialized here.
    # They are reset PER FILE inside the loop below — this is the fix.

    for pcap_path, label in tqdm(pcap_label_map.items(), desc="Processing PCAP files", unit="file"):
        try:
            pkts = rdpcap(pcap_path)
        except Exception as e:
            print(f"⚠️  Skipping {os.path.basename(pcap_path)}: {e}")
            continue

        # ── FIX: reset per file so timing never leaks across files/labels ──
        first_time = None
        last_time  = None

        for pkt in pkts:
            if not pkt.haslayer(TCP) or not pkt.haslayer(IP):
                continue

            tcp_layer = pkt[TCP]
            ip_layer  = pkt[IP]
            eth_layer = pkt[Ether] if pkt.haslayer(Ether) else None

            # ── DNP3 ports in dataset ──
            DNP3_PORTS = {20000, 20001, 20002, 20003}

            # ── Updated port filter in parse function ──
            if tcp_layer.sport not in DNP3_PORTS and tcp_layer.dport not in DNP3_PORTS:
                continue

            if not pkt.haslayer(Raw):
                continue

            raw = bytes(pkt[Raw])

            # ── DNP3 magic bytes check (0x05 0x64 = DNP3 start bytes) ──
            if len(raw) < 10 or raw[0] != 0x05 or raw[1] != 0x64:
                continue

            pkt_counter += 1
            if first_time is None:
                first_time = float(pkt.time)

            frame_time          = float(pkt.time)
            frame_time_relative = frame_time - first_time
            frame_time_delta    = None if last_time is None else frame_time - last_time
            last_time           = frame_time

            # ── Parse DNP3 Data Link Layer fields ──
            dnp3_len       = raw[2]                              # frame length
            dnp3_ctrl      = raw[3]                              # control byte
            dnp3_dst_addr  = int.from_bytes(raw[4:6], 'little') # destination address
            dnp3_src_addr  = int.from_bytes(raw[6:8], 'little') # source address
            dnp3_func      = raw[12] if len(raw) > 12 else None  # application function code
            dnp3_func_name = DNP3_FUNC_NAMES.get(dnp3_func, "UNKNOWN")

            # ── Control byte bit decomposition ──
            dnp3_dir       = (dnp3_ctrl & 0x80) >> 7  # DIR: direction bit
            dnp3_prm       = (dnp3_ctrl & 0x40) >> 6  # PRM: primary message
            dnp3_fcb       = (dnp3_ctrl & 0x20) >> 5  # FCB: frame count bit
            dnp3_fcv       = (dnp3_ctrl & 0x10) >> 4  # FCV: frame count valid
            dnp3_func_code = dnp3_ctrl & 0x0F          # link layer function code

            row = {
                # === File / Frame ===
                "source_file"          : os.path.basename(pcap_path).split(".")[0],
                "frame.number"         : pkt_counter,
                "frame.time_epoch"     : frame_time,
                "frame.time_relative"  : frame_time_relative,
                "frame.time_delta"     : frame_time_delta,
                "frame.len"            : len(pkt),
                "frame.cap_len"        : len(bytes(pkt)),

                # === Ethernet ===
                "eth.src"              : eth_layer.src if eth_layer else None,
                "eth.dst"              : eth_layer.dst if eth_layer else None,

                # === IP ===
                "ip.src"               : ip_layer.src,
                "ip.dst"               : ip_layer.dst,
                "ip.proto"             : ip_layer.proto,
                "ip.ttl"               : ip_layer.ttl,

                # === TCP ===
                "tcp.srcport"          : tcp_layer.sport,
                "tcp.dstport"          : tcp_layer.dport,
                "tcp.flags"            : tcp_layer.flags.value,
                "tcp.len"              : len(tcp_layer.payload),
                "tcp.window_size_value": tcp_layer.window,
                "tcp.checksum"         : tcp_layer.chksum,
                "tcp.time_delta"       : frame_time_delta,

                # === DNP3 Data Link Layer ===
                "dnp3_present"         : True,
                "dnp3.start"           : f"0x{raw[0]:02X}{raw[1]:02X}",
                "dnp3.len"             : dnp3_len,
                "dnp3.ctrl"            : dnp3_ctrl,
                "dnp3.dst_addr"        : dnp3_dst_addr,
                "dnp3.src_addr"        : dnp3_src_addr,
                "dnp3.dir"             : dnp3_dir,
                "dnp3.prm"             : dnp3_prm,
                "dnp3.fcb"             : dnp3_fcb,
                "dnp3.fcv"             : dnp3_fcv,
                "dnp3.func_code_link"  : dnp3_func_code,

                # === DNP3 Application Layer ===
                "dnp3.func_code"       : dnp3_func,
                "dnp3.func_name"       : dnp3_func_name,
                "dnp3.payload_len"     : len(raw),
                "dnp3.raw_hex"         : raw.hex()[:64],

                # === Label (ground truth from folder name) ===
                "label"                : label,
            }
            dnp3_rows.append(row)

    return pd.DataFrame(dnp3_rows)

## Step 4 — Label Mapping Function

Maps attack folder names from the IEEE Dataport dataset to standardized label strings. The folder name encodes the attack type, so this function auto-labels every PCAP file without manual annotation.

| Keyword in Folder | Label Assigned |
|---|---|
| `DISABLE` | `DISABLE_UNSOLICITED` |
| `COLD` | `COLD_RESTART` |
| `WARM` | `WARM_RESTART` |
| `MITM` | `MITM_DOS` |
| `ARP` | `ARP_POISONING` |
| `ENUMERATE` | `DNP3_ENUMERATE` |
| `INFO` | `DNP3_INFO` |
| `INITIALIZE` | `INIT_DATA` |
| `REPLAY` | `REPLAY` |
| `STOP` | `STOP_APP` |
| `NORMAL` | `NORMAL` |


In [4]:
def get_label(folder_name):
    """Map attack folder name to standardized label string."""
    f = folder_name.upper()
    if   "DISABLE"    in f: return "DISABLE_UNSOLICITED"
    elif "COLD"       in f: return "COLD_RESTART"
    elif "WARM"       in f: return "WARM_RESTART"
    elif "MITM"       in f: return "MITM_DOS"
    elif "ARP"        in f: return "ARP_POISONING"
    elif "ENUMERATE"  in f: return "DNP3_ENUMERATE"
    elif "INFO"       in f: return "DNP3_INFO"
    elif "INITIALIZE" in f: return "INIT_DATA"
    elif "REPLAY"     in f: return "REPLAY"
    elif "STOP"       in f: return "STOP_APP"
    elif "NORMAL"     in f: return "NORMAL"
    else:                   return folder_name


## Step 5 — Dataset Path Configuration

Set the root path of the IEEE Dataport DNP3 dataset on Google Drive.  
The dataset folder structure is:
```
DNP3_Intrusion_Detection_Dataset_Final/
├── 20200514_DNP3_Disable_Unsolicited_Messages_Attack/
│   ├── DNP3 PCAP Files/      ← USE THIS (filtered DNP3 only)
│   └── Total PCAP Files/     ← SKIP (all protocols, causes duplicates)
├── 20200515_DNP3_Cold_Restart_Attack/
│   ├── DNP3 PCAP Files/
│   └── Total PCAP Files/
└── ... (9 attack categories total)
```


In [9]:
data_path = "/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final"

## Step 6 — PCAP File Discovery

Recursively walk the dataset directory and build a `pcap_label_map` dictionary.

**Two critical filtering rules:**
1. **Skip `Total PCAP Files`** — these contain all protocol traffic (ARP, ICMP, TCP, etc.) and would duplicate records already in `DNP3 PCAP Files`
2. **Use only `DNP3 PCAP Files`** — pre-filtered by dataset authors to contain only DNP3 protocol packets

**Result:** `pcap_label_map = { full_path : attack_label }` with ~78 unique PCAP files across 9 attack categories.


In [10]:
# ── Only use "DNP3 PCAP Files" subfolder — skip "Total PCAP Files" ─────────────
pcap_label_map = {}

for root, dirs, files in os.walk(data_path):

    # ── Skip "Total PCAP Files" folders (duplicates) ──
    if "Total PCAP Files" in root:
        continue

    # ── Only process "DNP3 PCAP Files" folders ──
    if "DNP3 PCAP Files" not in root:
        continue

    for file in files:
        if file.endswith(".pcap") or file.endswith(".pcapng"):
            full_path     = os.path.join(root, file)

            # Label from parent attack folder (2 levels up from "DNP3 PCAP Files")
            attack_folder = os.path.basename(os.path.dirname(root))
            label         = get_label(attack_folder)

            pcap_label_map[full_path] = label

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total unique PCAP files : {len(pcap_label_map)}")
print()

from collections import Counter
label_counts = Counter(pcap_label_map.values())
print("Files per label:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:<25} : {count} files")

print()
print("Sample entries:")
for path, label in list(pcap_label_map.items())[:5]:
    print(f"  [{label}]  {os.path.basename(path)}")


Total unique PCAP files : 78

Files per label:
  20200516_DNP3_Ιnfo        : 7 files
  COLD_RESTART              : 12 files
  DISABLE_UNSOLICITED       : 12 files
  DNP3_ENUMERATE            : 7 files
  INIT_DATA                 : 7 files
  MITM_DOS                  : 7 files
  REPLAY                    : 7 files
  STOP_APP                  : 7 files
  WARM_RESTART              : 12 files

Sample entries:
  [COLD_RESTART]  20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_01.pcap
  [COLD_RESTART]  20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap
  [COLD_RESTART]  20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_03.pcap
  [COLD_RESTART]  20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Master.pcap
  [COLD_RESTART]  20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_01.pcap


## Step 7 — Verify DNP3 Ports in Dataset

Before parsing all files, check which TCP ports DNP3 traffic actually uses in this dataset.  

The standard DNP3 port is `20000`, but this dataset's testbed configured multiple slave nodes on different ports (`20000`, `20001`, `20002`, `20003`).  
This is critical — if we only filter on port `20000`, we would miss most packets.


In [11]:
# Check all TCP ports used in first 5 PCAP files
from scapy.all import rdpcap, TCP

all_ports = set()
for pcap_path in list(pcap_label_map.keys())[:5]:
    pkts = rdpcap(pcap_path)
    for pkt in pkts:
        if pkt.haslayer(TCP):
            all_ports.add(pkt[TCP].dport)
            all_ports.add(pkt[TCP].sport)

# Filter only DNP3 port range
dnp3_ports = {p for p in all_ports if 19999 < p < 20010}
print("DNP3 ports found in dataset:", dnp3_ports)


DNP3 ports found in dataset: {20000, 20001, 20002, 20003}


## Step 8 — Sample Selection (2 Files Per Label)

Instead of processing all ~78 PCAP files at once (slow in Colab), select **2 files per attack label** for an initial balanced extraction run.  
This gives representative coverage of all 9 attack categories for testing and validation before running the full dataset.


In [27]:
from collections import defaultdict

# ── Get 2 files per label ──────────────────────────────────────────────────────
label_groups = defaultdict(list)

for path, label in pcap_label_map.items():
    label_groups[label].append(path)

# Take only 2 files per label
pcap_label_map_sample = {}
for label, paths in label_groups.items():
    for path in paths[:2]:
        pcap_label_map_sample[path] = label

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total files (sample) : {len(pcap_label_map_sample)}")
print()
for label, paths in label_groups.items():
    selected = [os.path.basename(p) for p in paths]
    print(f"  {label:<25} → {selected}")


Total files (sample) : 18

  COLD_RESTART              → ['20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_01.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_03.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Master.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_01.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_02.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_03.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_04.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_05.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_06.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_07.pcap', '20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_08.pcap']
  DNP3_ENUMERATE            → ['20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attacker_01.pcap', '20200516_DNP3

In [28]:
len(pcap_label_map_sample)

18

In [29]:
for root, dirs, files in os.walk(data_path):
    if "Total PCAP Files" in root:
        continue
    if "DNP3 PCAP Files" not in root:
        continue

    for file in files:
        if file.endswith(".pcap") or file.endswith(".pcapng"):
            # ── FIX: only use Attacker files for attack labels ──
            if "Attacker" not in file:
                continue

            full_path     = os.path.join(root, file)
            attack_folder = os.path.basename(os.path.dirname(root))
            label         = get_label(attack_folder)
            pcap_label_map[full_path] = label

In [30]:
len(pcap_label_map)

78

In [31]:
pcap_label_map

{'/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_01.pcap': 'COLD_RESTART',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap': 'COLD_RESTART',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_03.pcap': 'COLD_RESTART',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Master.pcap': 'COLD_RESTART',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Col

In [32]:
# ── Keep only Attacker files for attack-label training ──
pcap_label_map_attackers_only = {
    path: label
    for path, label in pcap_label_map.items()
    if "Attacker" in os.path.basename(path)
}

print(f"Filtered from {len(pcap_label_map)} → {len(pcap_label_map_attackers_only)} files")

from collections import Counter
label_counts = Counter(pcap_label_map_attackers_only.values())
print("\nFiles per label (Attacker-only):")
for label, count in sorted(label_counts.items()):
    print(f"  {label:<25} : {count} files")

Filtered from 78 → 27 files

Files per label (Attacker-only):
  20200516_DNP3_Ιnfo        : 3 files
  COLD_RESTART              : 3 files
  DISABLE_UNSOLICITED       : 3 files
  DNP3_ENUMERATE            : 3 files
  INIT_DATA                 : 3 files
  MITM_DOS                  : 3 files
  REPLAY                    : 3 files
  STOP_APP                  : 3 files
  WARM_RESTART              : 3 files


In [33]:
pcap_label_map_attackers_only

{'/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_01.pcap': 'COLD_RESTART',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_02.pcap': 'COLD_RESTART',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200515_DNP3_Cold_Restart_Attack/DNP3 PCAP Files/20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Attacker_03.pcap': 'COLD_RESTART',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200516_DNP3_Enumerate/DNP3 PCAP Files/20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attacker_01.pcap': 'DNP3_ENUMERATE',
 '/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/20200516_DNP3_Enumerate/DNP3 PCAP Files/20200516_DNP3_Enumerate_UOWM_DNP3_Datase

## Step 9 — Run Feature Extraction

Run `parse_pcap_files()` on the sampled PCAP files.  
Each valid DNP3 packet becomes one row in the output DataFrame with 31 feature columns.


In [34]:
# ── Run parser on sampled files ───────────────────────────────────────────────
df = parse_pcap_files(pcap_label_map_attackers_only)

Processing PCAP files:   0%|          | 0/27 [00:00<?, ?file/s]

## Step 10 — Extraction Summary

Verify the extraction results — total packets, label distribution, shape, and column list.


In [35]:
print(f"Total packets extracted : {len(df)}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nShape: {df.shape}")
print(f"\nColumns:")
for col in df.columns.tolist():
    print(f"  {col}")

Total packets extracted : 1251551

Label distribution:
label
DISABLE_UNSOLICITED    382988
WARM_RESTART           382912
COLD_RESTART           382891
INIT_DATA               45245
STOP_APP                45242
REPLAY                   4282
DNP3_ENUMERATE           3375
20200516_DNP3_Ιnfo       3342
MITM_DOS                 1274
Name: count, dtype: int64

Shape: (1251551, 36)

Columns:
  source_file
  frame.number
  frame.time_epoch
  frame.time_relative
  frame.time_delta
  frame.len
  frame.cap_len
  eth.src
  eth.dst
  ip.src
  ip.dst
  ip.proto
  ip.ttl
  tcp.srcport
  tcp.dstport
  tcp.flags
  tcp.len
  tcp.window_size_value
  tcp.checksum
  tcp.time_delta
  dnp3_present
  dnp3.start
  dnp3.len
  dnp3.ctrl
  dnp3.dst_addr
  dnp3.src_addr
  dnp3.dir
  dnp3.prm
  dnp3.fcb
  dnp3.fcv
  dnp3.func_code_link
  dnp3.func_code
  dnp3.func_name
  dnp3.payload_len
  dnp3.raw_hex
  label


## Step 11 — Preview Extracted Data

Show first 5 rows of the extracted DataFrame to verify features are correctly parsed.


In [36]:
df.head()


,source_file,frame.number,frame.time_epoch,frame.time_relative,frame.time_delta,frame.len,frame.cap_len,eth.src,eth.dst,ip.src,ip.dst,ip.proto,ip.ttl,tcp.srcport,tcp.dstport,tcp.flags,tcp.len,tcp.window_size_value,tcp.checksum,tcp.time_delta,dnp3_present,dnp3.start,dnp3.len,dnp3.ctrl,dnp3.dst_addr,dnp3.src_addr,dnp3.dir,dnp3.prm,dnp3.fcb,dnp3.fcv,dnp3.func_code_link,dnp3.func_code,dnp3.func_name,dnp3.payload_len,dnp3.raw_hex,label
0,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,1,1.589556e+09,0.000000,NaN,84,84,be:0a:53:35:94:71,be:0a:53:15:bc:f6,192.168.1.1,192.168.1.5,6,64,41538,20001,24,18,312,24389,NaN,True,0x0564,11,196,13,2,1,1,0,0,4,1.0,READ,18,05640bc40d0002006db7cdcc013c0206505b,COLD_RESTART
1,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,2,1.589556e+09,0.000756,0.000756,83,83,be:0a:53:15:bc:f6,be:0a:53:35:94:71,192.168.1.5,192.168.1.1,6,64,20001,41538,24,17,227,7867,0.000756,True,0x0564,10,68,2,13,0,1,0,0,4,129.0,RESPONSE,17,05640a4402000d004ea8d2cc810000062d,COLD_RESTART
2,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,3,1.589556e+09,0.000836,0.000080,84,84,be:0a:53:35:94:71,be:0a:53:07:a6:60,192.168.1.1,192.168.1.9,6,64,51971,20001,24,18,312,49036,0.000080,True,0x0564,11,196,17,2,1,1,0,0,4,1.0,READ,18,05640bc4110002004e39cdcc013c0206505b,COLD_RESTART
3,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,4,1.589556e+09,0.001856,0.001020,84,84,be:0a:53:35:94:71,be:0a:50:6c:87:fa,192.168.1.1,192.168.1.4,6,64,45297,20001,24,18,312,56407,0.001020,True,0x0564,11,196,12,2,1,1,0,0,4,1.0,READ,18,05640bc40c0002008575cdcc013c0206505b,COLD_RESTART
4,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,5,1.589556e+09,0.002557,0.000701,83,83,be:0a:50:6c:87:fa,be:0a:53:35:94:71,192.168.1.4,192.168.1.1,6,64,20001,45297,24,17,227,561,0.000701,True,0x0564,10,68,2,12,0,1,0,0,4,129.0,RESPONSE,17,05640a4402000c000003d2cc810000062d,COLD_RESTART


In [37]:
df.columns.tolist()

['source_file',
 'frame.number',
 'frame.time_epoch',
 'frame.time_relative',
 'frame.time_delta',
 'frame.len',
 'frame.cap_len',
 'eth.src',
 'eth.dst',
 'ip.src',
 'ip.dst',
 'ip.proto',
 'ip.ttl',
 'tcp.srcport',
 'tcp.dstport',
 'tcp.flags',
 'tcp.len',
 'tcp.window_size_value',
 'tcp.checksum',
 'tcp.time_delta',
 'dnp3_present',
 'dnp3.start',
 'dnp3.len',
 'dnp3.ctrl',
 'dnp3.dst_addr',
 'dnp3.src_addr',
 'dnp3.dir',
 'dnp3.prm',
 'dnp3.fcb',
 'dnp3.fcv',
 'dnp3.func_code_link',
 'dnp3.func_code',
 'dnp3.func_name',
 'dnp3.payload_len',
 'dnp3.raw_hex',
 'label']

## Step 12 — Save to Google Drive

Create output directory on Google Drive and save the extracted features as CSV.  



In [38]:
import os

directory_path = '/content/drive/MyDrive/SytheticDataWatersDirectory'
os.makedirs(directory_path, exist_ok=True)
print(f"Output directory: {directory_path}")


Output directory: /content/drive/MyDrive/SytheticDataWatersDirectory


In [39]:
df.to_csv(f"{directory_path}/my_data_dnp3_update.csv", index=False)
print(f"Saved: {directory_path}/my_data_dnp3_update.csv")
print(f"Shape: {df.shape}")


Saved: /content/drive/MyDrive/SytheticDataWatersDirectory/my_data_dnp3_update.csv
Shape: (1251551, 36)


## Step 13 — Reload & Verify

Reload the saved CSV and verify shape to confirm the file was saved correctly.


In [40]:
df = pd.read_csv(f"{directory_path}/my_data_dnp3_update.csv")
print(f"Reloaded shape: {df.shape}")
df.head()


Reloaded shape: (1251551, 36)


,source_file,frame.number,frame.time_epoch,frame.time_relative,frame.time_delta,frame.len,frame.cap_len,eth.src,eth.dst,ip.src,ip.dst,ip.proto,ip.ttl,tcp.srcport,tcp.dstport,tcp.flags,tcp.len,tcp.window_size_value,tcp.checksum,tcp.time_delta,dnp3_present,dnp3.start,dnp3.len,dnp3.ctrl,dnp3.dst_addr,dnp3.src_addr,dnp3.dir,dnp3.prm,dnp3.fcb,dnp3.fcv,dnp3.func_code_link,dnp3.func_code,dnp3.func_name,dnp3.payload_len,dnp3.raw_hex,label
0,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,1,1.589556e+09,0.000000,NaN,84,84,be:0a:53:35:94:71,be:0a:53:15:bc:f6,192.168.1.1,192.168.1.5,6,64,41538,20001,24,18,312,24389,NaN,True,0x0564,11,196,13,2,1,1,0,0,4,1.0,READ,18,05640bc40d0002006db7cdcc013c0206505b,COLD_RESTART
1,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,2,1.589556e+09,0.000756,0.000756,83,83,be:0a:53:15:bc:f6,be:0a:53:35:94:71,192.168.1.5,192.168.1.1,6,64,20001,41538,24,17,227,7867,0.000756,True,0x0564,10,68,2,13,0,1,0,0,4,129.0,RESPONSE,17,05640a4402000d004ea8d2cc810000062d,COLD_RESTART
2,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,3,1.589556e+09,0.000836,0.000080,84,84,be:0a:53:35:94:71,be:0a:53:07:a6:60,192.168.1.1,192.168.1.9,6,64,51971,20001,24,18,312,49036,0.000080,True,0x0564,11,196,17,2,1,1,0,0,4,1.0,READ,18,05640bc4110002004e39cdcc013c0206505b,COLD_RESTART
3,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,4,1.589556e+09,0.001856,0.001020,84,84,be:0a:53:35:94:71,be:0a:50:6c:87:fa,192.168.1.1,192.168.1.4,6,64,45297,20001,24,18,312,56407,0.001020,True,0x0564,11,196,12,2,1,1,0,0,4,1.0,READ,18,05640bc40c0002008575cdcc013c0206505b,COLD_RESTART
4,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,5,1.589556e+09,0.002557,0.000701,83,83,be:0a:50:6c:87:fa,be:0a:53:35:94:71,192.168.1.4,192.168.1.1,6,64,20001,45297,24,17,227,561,0.000701,True,0x0564,10,68,2,12,0,1,0,0,4,129.0,RESPONSE,17,05640a4402000c000003d2cc810000062d,COLD_RESTART


In [42]:
df.groupby(['source_file','label'])[['frame.time_relative','tcp.time_delta']].agg(['min','max'])

frame.time_relative  \
                                                                                       min   
source_file                                        label                                     
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED                 0.0   
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED                 0.0   
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED                 0.0   
20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dat... COLD_RESTART                        0.0   
20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dat... COLD_RESTART                        0.0   
20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dat... COLD_RESTART                        0.0   
20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dat... WARM_RESTART                        0.0   
20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dat... WARM_RESTART                        0.0   
20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dat... WARM_RESTART                        0.0   
20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attac... DNP3_ENUMERATE                      0.0   
20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attac... DNP3_ENUMERATE                      0.0   
20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attac... DNP3_ENUMERATE                      0.0   
20200516_DNP3_info_UOWM_DNP3_Dataset_Attacker_01   20200516_DNP3_Ιnfo                  0.0   
20200516_DNP3_info_UOWM_DNP3_Dataset_Attacker_02   20200516_DNP3_Ιnfo                  0.0   
20200516_DNP3_info_UOWM_DNP3_Dataset_Attacker_03   20200516_DNP3_Ιnfo                  0.0   
20200518_Initialize_Data_Attack_UOWM_DNP3_Datas... INIT_DATA                           0.0   
20200518_Initialize_Data_Attack_UOWM_DNP3_Datas... INIT_DATA                           0.0   
20200518_Initialize_Data_Attack_UOWM_DNP3_Datas... INIT_DATA                           0.0   
20200518_MITM_DOS_UOWM_DNP3_Dataset_Attacker_01    MITM_DOS                            0.0   
20200518_MITM_DOS_UOWM_DNP3_Dataset_Attacker_02    MITM_DOS                            0.0   
20200518_MITM_DOS_UOWM_DNP3_Dataset_Attacker_03    MITM_DOS                            0.0   
20200518_Replay_UOWM_DNP3_Dataset_Attacker_01      REPLAY                              0.0   
20200518_Replay_UOWM_DNP3_Dataset_Attacker_02      REPLAY                              0.0   
20200518_Replay_UOWM_DNP3_Dataset_Attacker_03      REPLAY                              0.0   
20200519_Stop_Application_Attack_UOWM_DNP3_Data... STOP_APP                            0.0   
20200519_Stop_Application_Attack_UOWM_DNP3_Data... STOP_APP                            0.0   
20200519_Stop_Application_Attack_UOWM_DNP3_Data... STOP_APP                            0.0   

                                                                                      \
                                                                                 max   
source_file                                        label                               
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED  14397.917178   
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED  14398.161433   
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED  14399.080072   
20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dat... COLD_RESTART         14397.532772   
20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dat... COLD_RESTART         14398.026292   
20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dat... COLD_RESTART         14397.873743   
20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dat... WARM_RESTART         14397.473986   
20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dat... WARM_RESTART         14398.931632   
20200515_DNP3_Warm_Restart_Attack_UOWM_DNP3_Dat... WARM_RESTART         14399.179668   
20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attac... DNP3_ENUMERATE       14324.408243   
20200516_DNP3_Enumerate_UOWM_DNP3_Dataset_Attac... DNP3_ENUMERATE       14347.750986   
20200516_DNP3_Enum

In [45]:
display(pd.crosstab(df['label'], df['dnp3.func_code']))
display(pd.crosstab(df['label'], df['dnp3.func_name']))

dnp3.func_code,0.0,1.0,5.0,11.0,13.0,14.0,15.0,18.0,21.0,129.0,130.0,161.0
label,,,,,,,,,,,,
20200516_DNP3_Ιnfo,0,0,1671,0,0,0,0,0,0,0,1671,0
COLD_RESTART,5755,178464,0,0,10104,0,0,0,0,188568,0,0
DISABLE_UNSOLICITED,5760,178478,0,0,0,0,0,0,10136,188614,0,0
DNP3_ENUMERATE,0,0,1685,0,0,0,0,0,0,0,1690,0
INIT_DATA,680,21196,0,0,0,0,1117,0,0,22252,0,0
MITM_DOS,42,727,0,2,0,0,0,0,0,500,0,1
REPLAY,250,2151,0,0,0,0,0,0,0,1881,0,0
STOP_APP,677,21242,0,0,0,0,0,1069,0,22254,0,0
WARM_RESTART,5760,178472,0,0,0,10104,0,0,0,188576,0,0


dnp3.func_name,COLD_RESTART,CONFIRM,DIRECT_OPERATE,DISABLE_UNSOLICITED,FREEZE_AT_TIME,INITIALIZE_DATA,READ,RESPONSE,STOP_APPL,UNKNOWN,UNSOLICITED_RESPONSE,WARM_RESTART
label,,,,,,,,,,,,
20200516_DNP3_Ιnfo,0,0,1671,0,0,0,0,0,0,0,1671,0
COLD_RESTART,10104,5755,0,0,0,0,178464,188568,0,0,0,0
DISABLE_UNSOLICITED,0,5760,0,10136,0,0,178478,188614,0,0,0,0
DNP3_ENUMERATE,0,0,1685,0,0,0,0,0,0,0,1690,0
INIT_DATA,0,680,0,0,0,1117,21196,22252,0,0,0,0
MITM_DOS,0,42,0,0,2,0,727,500,0,3,0,0
REPLAY,0,250,0,0,0,0,2151,1881,0,0,0,0
STOP_APP,0,677,0,0,0,0,21242,22254,1069,0,0,0
WARM_RESTART,0,5760,0,0,0,0,178472,188576,0,0,0,10104


In [46]:
print(df.groupby('label')[['tcp.srcport','tcp.dstport']].nunique())
for lbl in df.label.unique():
    print(lbl, sorted(df[df.label==lbl]['tcp.dstport'].unique())[:10])

                     tcp.srcport  tcp.dstport
label                                        
20200516_DNP3_Ιnfo          1196         1196
COLD_RESTART                  27           27
DISABLE_UNSOLICITED           27           27
DNP3_ENUMERATE              1166         1168
INIT_DATA                      6            6
MITM_DOS                       8            5
REPLAY                         4            4
STOP_APP                       6            6
WARM_RESTART                  27           27
COLD_RESTART [np.int64(20001), np.int64(20002), np.int64(20003), np.int64(33180), np.int64(36519), np.int64(37363), np.int64(39338), np.int64(40709), np.int64(41538), np.int64(42312)]
DNP3_ENUMERATE [np.int64(20000), np.int64(36310), np.int64(36314), np.int64(36318), np.int64(36322), np.int64(36326), np.int64(36330), np.int64(36334), np.int64(36338), np.int64(36342)]
WARM_RESTART [np.int64(20001), np.int64(20002), np.int64(20003), np.int64(32935), np.int64(34770), np.int64(36435), np.int64

In [47]:
print(df.duplicated(subset=['dnp3.raw_hex','frame.time_epoch']).sum())
print(df.duplicated().sum())

0
0


In [51]:
print(df.groupby('label')[['ip.src','ip.dst']].nunique())

                     ip.src  ip.dst
label                              
20200516_DNP3_Ιnfo        6       6
COLD_RESTART             11      11
DISABLE_UNSOLICITED      11      11
DNP3_ENUMERATE            6       6
INIT_DATA                 6       6
MITM_DOS                  4       4
REPLAY                    4       4
STOP_APP                  6       6
WARM_RESTART             11      11


In [53]:

DNP3_PORTS = {20000, 20001, 20002, 20003}

def run_diagnostics(df: pd.DataFrame):
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 250)

    print("="*90)
    print("1. TIMING SANITY  (frame.time_relative / tcp.time_delta)")
    print("="*90)
    tg = df.groupby(['source_file','label'])[['frame.time_relative','tcp.time_delta']].agg(['min','max'])
    print(tg)
    bad_relative = df[df['frame.time_relative'] < 0]
    bad_delta_neg = df[df['tcp.time_delta'] < 0]
    bad_delta_big = df[df['tcp.time_delta'] > 300]   # >5 min between packets is suspicious for these attacks
    print(f"\nRows with NEGATIVE frame.time_relative : {len(bad_relative)}")
    print(f"Rows with NEGATIVE tcp.time_delta       : {len(bad_delta_neg)}")
    print(f"Rows with tcp.time_delta > 300s          : {len(bad_delta_big)}")

    print("\n" + "="*90)
    print("2. LABEL vs FUNC_CODE CONSISTENCY")
    print("="*90)
    print(pd.crosstab(df['label'], df['dnp3.func_name']))

    print("\n" + "="*90)
    print("3. PORT CONSISTENCY  (must be one of 20000/20001/20002/20003)")
    print("="*90)
    for lbl in sorted(df['label'].unique()):
        sub = df[df['label'] == lbl]
        srcports = set(sub['tcp.srcport'])
        dstports = set(sub['tcp.dstport'])
        fixed_seen = (srcports | dstports) & DNP3_PORTS
        non_dnp3_rows = sub[~(sub['tcp.srcport'].isin(DNP3_PORTS) | sub['tcp.dstport'].isin(DNP3_PORTS))]

        print(f"{lbl:<22} fixed_ports_seen={str(sorted(fixed_seen)):<25} "
              f"rows_with_NO_dnp3_port={len(non_dnp3_rows)}")

    # breakdown of which specific DNP3 port each label actually uses, and how often
    print("\n--- Per-label DNP3 port usage breakdown (count of rows per port) ---")
    port_rows = []
    for lbl in sorted(df['label'].unique()):
        sub = df[df['label'] == lbl]
        for p in sorted(DNP3_PORTS):
            cnt = ((sub['tcp.srcport'] == p) | (sub['tcp.dstport'] == p)).sum()
            port_rows.append({'label': lbl, 'port': p, 'row_count': cnt})
    port_df = pd.DataFrame(port_rows)
    print(port_df.pivot(index='label', columns='port', values='row_count').fillna(0).astype(int))

    print("\n" + "="*90)
    print("4. DUPLICATE ROWS")
    print("="*90)
    dup_hex_time = df.duplicated(subset=['dnp3.raw_hex','frame.time_epoch']).sum()
    dup_full = df.duplicated().sum()
    print(f"Duplicate (raw_hex + time_epoch) rows : {dup_hex_time}")
    print(f"Fully duplicate rows                  : {dup_full}")

    print("\n" + "="*90)
    print("5. MISSING VALUES")
    print("="*90)
    nulls = df.isnull().sum()
    print(nulls[nulls > 0])
    print(f"\nRows with null dnp3.func_code: {df['dnp3.func_code'].isna().sum()} "
          f"({df['dnp3.func_code'].isna().mean()*100:.1f}%)")

    print("\n" + "="*90)
    print("6. SIZE / OUTLIER CHECK")
    print("="*90)
    print(df[['frame.len','frame.cap_len','tcp.len','tcp.window_size_value','dnp3.payload_len']].describe())
    truncated = df[df['frame.len'] != df['frame.cap_len']]
    print(f"\nRows where frame.len != frame.cap_len (possible truncation): {len(truncated)}")

    print("\n" + "="*90)
    print("7. CLASS BALANCE")
    print("="*90)
    print(df['label'].value_counts())
    print("\nRows per source_file (grouped by label):")
    print(df.groupby(['label','source_file']).size())

    print("\n" + "="*90)
    print("8. IP SANITY PER LABEL")
    print("="*90)
    print(df.groupby('label')[['ip.src','ip.dst']].nunique())
    for lbl in sorted(df['label'].unique()):
        sub = df[df['label']==lbl]
        print(f"{lbl:<22} ip.src={sorted(sub['ip.src'].unique())}  ip.dst={sorted(sub['ip.dst'].unique())}")

    print("\n" + "="*90)
    print("DIAGNOSTIC COMPLETE")
    print("="*90)


if __name__ == "__main__":
    run_diagnostics(df)

1. TIMING SANITY  (frame.time_relative / tcp.time_delta)
                                                                       frame.time_relative               tcp.time_delta             
                                                                                       min           max            min          max
source_file                                        label                                                                            
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED                 0.0  14397.917178      -0.000003     1.902170
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED                 0.0  14398.161433      -0.000009     2.000240
20200514_DNP3_Disable_Unsolicited_Messages_Atta... DISABLE_UNSOLICITED                 0.0  14399.080072      -0.000002     1.927444
20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dat... COLD_RESTART                        0.0  14397.532772      -0.000040     1.944457
20200515_DNP

In [55]:
df['frame.time_delta'] = df.groupby('source_file')['frame.time_delta'].transform(lambda x: x.ffill())
df['tcp.time_delta'] = df.groupby('source_file')['tcp.time_delta'].transform(lambda x: x.ffill())

In [56]:
df = df.dropna(subset=['dnp3.func_code'])

In [57]:
df.shape

(1251549, 36)

In [58]:
DNP3_PORTS = {20000, 20001, 20002, 20003}

df = df[
    df['tcp.srcport'].isin(DNP3_PORTS) |
    df['tcp.dstport'].isin(DNP3_PORTS)
]

print(f"Rows after filtering: {len(df)}")

Rows after filtering: 1251549


In [59]:
# Check no non-DNP3 ports slipped through
print(df['tcp.srcport'].unique())
print(df['tcp.dstport'].unique())

[41538 20001 51971 ... 42776 57231 54309]
[20001 41538 45297 ... 42776 57231 54309]


In [60]:
# Should only show DNP3 ports
dnp3_ports_seen = set(df['tcp.srcport'].unique()) | set(df['tcp.dstport'].unique())
non_dnp3 = dnp3_ports_seen - DNP3_PORTS
print(f"Non-DNP3 ports still present: {non_dnp3}")

Non-DNP3 ports still present: {np.int64(40960), np.int64(40962), np.int64(40964), np.int64(40966), np.int64(40968), np.int64(40970), np.int64(40972), np.int64(40974), np.int64(40976), np.int64(40978), np.int64(40980), np.int64(40982), np.int64(40984), np.int64(40986), np.int64(40988), np.int64(40990), np.int64(40992), np.int64(40994), np.int64(40996), np.int64(40998), np.int64(41000), np.int64(41002), np.int64(41004), np.int64(41006), np.int64(41008), np.int64(41010), np.int64(41012), np.int64(41014), np.int64(41016), np.int64(41018), np.int64(41020), np.int64(41022), np.int64(41024), np.int64(41026), np.int64(41028), np.int64(41030), np.int64(41032), np.int64(41034), np.int64(41036), np.int64(41038), np.int64(41040), np.int64(41042), np.int64(41044), np.int64(41046), np.int64(41048), np.int64(41050), np.int64(41052), np.int64(41054), np.int64(41056), np.int64(41058), np.int64(41060), np.int64(41062), np.int64(41064), np.int64(41065), np.int64(41066), np.int64(41068), np.int64(41070), 

In [61]:
# Both sides DNP3 at the same time? Should be 0 or very rare
both = df[df['tcp.srcport'].isin(DNP3_PORTS) & df['tcp.dstport'].isin(DNP3_PORTS)]
print(f"Rows with DNP3 on BOTH sides: {len(both)}")

# DNP3 on src side only
src_only = df[df['tcp.srcport'].isin(DNP3_PORTS) & ~df['tcp.dstport'].isin(DNP3_PORTS)]
print(f"DNP3 on src only (server→client): {len(src_only)}")

# DNP3 on dst side only
dst_only = df[~df['tcp.srcport'].isin(DNP3_PORTS) & df['tcp.dstport'].isin(DNP3_PORTS)]
print(f"DNP3 on dst only (client→server): {len(dst_only)}")

Rows with DNP3 on BOTH sides: 0
DNP3 on src only (server→client): 634931
DNP3 on dst only (client→server): 616618


In [62]:
DNP3_PORTS = {20000, 20001, 20002, 20003}

df['direction'] = df['tcp.srcport'].apply(
    lambda x: 'response' if x in DNP3_PORTS else 'request'
)

print(df['direction'].value_counts())

direction
response    634931
request     616618
Name: count, dtype: int64


In [63]:
df.to_csv(f"{directory_path}/my_data_dnp3_update_clean_data.csv", index=False)
print(f"Saved: {directory_path}/my_data_dnp3_update_clean_data.csv")

Saved: /content/drive/MyDrive/SytheticDataWatersDirectory/my_data_dnp3_update_clean_data.csv


In [64]:
df = pd.read_csv(f"{directory_path}/my_data_dnp3_update_clean_data.csv")
print(f"Reloaded shape: {df.shape}")
df.head()


Reloaded shape: (1251549, 37)


,source_file,frame.number,frame.time_epoch,frame.time_relative,frame.time_delta,frame.len,frame.cap_len,eth.src,eth.dst,ip.src,ip.dst,ip.proto,ip.ttl,tcp.srcport,tcp.dstport,tcp.flags,tcp.len,tcp.window_size_value,tcp.checksum,tcp.time_delta,dnp3_present,dnp3.start,dnp3.len,dnp3.ctrl,dnp3.dst_addr,dnp3.src_addr,dnp3.dir,dnp3.prm,dnp3.fcb,dnp3.fcv,dnp3.func_code_link,dnp3.func_code,dnp3.func_name,dnp3.payload_len,dnp3.raw_hex,label,direction
0,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,1,1.589556e+09,0.000000,NaN,84,84,be:0a:53:35:94:71,be:0a:53:15:bc:f6,192.168.1.1,192.168.1.5,6,64,41538,20001,24,18,312,24389,NaN,True,0x0564,11,196,13,2,1,1,0,0,4,1.0,READ,18,05640bc40d0002006db7cdcc013c0206505b,COLD_RESTART,request
1,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,2,1.589556e+09,0.000756,0.000756,83,83,be:0a:53:15:bc:f6,be:0a:53:35:94:71,192.168.1.5,192.168.1.1,6,64,20001,41538,24,17,227,7867,0.000756,True,0x0564,10,68,2,13,0,1,0,0,4,129.0,RESPONSE,17,05640a4402000d004ea8d2cc810000062d,COLD_RESTART,response
2,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,3,1.589556e+09,0.000836,0.000080,84,84,be:0a:53:35:94:71,be:0a:53:07:a6:60,192.168.1.1,192.168.1.9,6,64,51971,20001,24,18,312,49036,0.000080,True,0x0564,11,196,17,2,1,1,0,0,4,1.0,READ,18,05640bc4110002004e39cdcc013c0206505b,COLD_RESTART,request
3,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,4,1.589556e+09,0.001856,0.001020,84,84,be:0a:53:35:94:71,be:0a:50:6c:87:fa,192.168.1.1,192.168.1.4,6,64,45297,20001,24,18,312,56407,0.001020,True,0x0564,11,196,12,2,1,1,0,0,4,1.0,READ,18,05640bc40c0002008575cdcc013c0206505b,COLD_RESTART,request
4,20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Da...,5,1.589556e+09,0.002557,0.000701,83,83,be:0a:50:6c:87:fa,be:0a:53:35:94:71,192.168.1.4,192.168.1.1,6,64,20001,45297,24,17,227,561,0.000701,True,0x0564,10,68,2,12,0,1,0,0,4,129.0,RESPONSE,17,05640a4402000c000003d2cc810000062d,COLD_RESTART,response


# **User Prompt**

In [ ]:
!pip install scapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 103.9 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from scapy.all import rdpcap, TCP, IP, Ether, Raw

# ── DNP3 Function Code Names (per DNP3 spec) ──────────────────────────────────
DNP3_FUNC_NAMES = {
    0: "CONFIRM", 1: "READ", 2: "WRITE", 3: "SELECT",
    4: "OPERATE", 5: "DIRECT_OPERATE", 6: "DIRECT_OPERATE_NR",
    7: "IMMED_FREEZE", 8: "IMMED_FREEZE_NR", 9: "FREEZE_CLEAR",
    10: "FREEZE_CLEAR_NR", 11: "FREEZE_AT_TIME", 12: "FREEZE_AT_TIME_NR",
    13: "COLD_RESTART", 14: "WARM_RESTART", 15: "INITIALIZE_DATA",
    16: "INITIALIZE_APPL", 17: "START_APPL", 18: "STOP_APPL",
    19: "SAVE_CONFIG", 20: "ENABLE_UNSOLICITED", 21: "DISABLE_UNSOLICITED",
    22: "ASSIGN_CLASS", 23: "DELAY_MEASURE", 24: "RECORD_CURRENT_TIME",
    129: "RESPONSE", 130: "UNSOLICITED_RESPONSE",
}


def parse_pcap_files(pcap_label_map: dict) -> pd.DataFrame:
    """
    Parse all PCAP files in pcap_label_map and extract DNP3 packet-level features.

    Args:
        pcap_label_map: dict of {full_pcap_path: attack_label}

    Returns:
        pd.DataFrame with 31 feature columns + label
    """
    dnp3_rows   = []
    pkt_counter = 0
    # NOTE: first_time / last_time are no longer initialized here.
    # They are reset PER FILE inside the loop below — this is the fix.

    for pcap_path, label in tqdm(pcap_label_map.items(), desc="Processing PCAP files", unit="file"):
        try:
            pkts = rdpcap(pcap_path)
        except Exception as e:
            print(f"⚠️  Skipping {os.path.basename(pcap_path)}: {e}")
            continue

        # ── FIX: reset per file so timing never leaks across files/labels ──
        first_time = None
        last_time  = None

        for pkt in pkts:
            if not pkt.haslayer(TCP) or not pkt.haslayer(IP):
                continue

            tcp_layer = pkt[TCP]
            ip_layer  = pkt[IP]
            eth_layer = pkt[Ether] if pkt.haslayer(Ether) else None

            # ── DNP3 ports in dataset ──
            DNP3_PORTS = {20000, 20001, 20002, 20003}

            # ── Updated port filter in parse function ──
            if tcp_layer.sport not in DNP3_PORTS and tcp_layer.dport not in DNP3_PORTS:
                continue

            if not pkt.haslayer(Raw):
                continue

            raw = bytes(pkt[Raw])

            # ── DNP3 magic bytes check (0x05 0x64 = DNP3 start bytes) ──
            if len(raw) < 10 or raw[0] != 0x05 or raw[1] != 0x64:
                continue

            pkt_counter += 1
            if first_time is None:
                first_time = float(pkt.time)

            frame_time          = float(pkt.time)
            frame_time_relative = frame_time - first_time
            frame_time_delta    = None if last_time is None else frame_time - last_time
            last_time           = frame_time

            # ── Parse DNP3 Data Link Layer fields ──
            dnp3_len       = raw[2]                              # frame length
            dnp3_ctrl      = raw[3]                              # control byte
            dnp3_dst_addr  = int.from_bytes(raw[4:6], 'little') # destination address
            dnp3_src_addr  = int.from_bytes(raw[6:8], 'little') # source address
            dnp3_func      = raw[12] if len(raw) > 12 else None  # application function code
            dnp3_func_name = DNP3_FUNC_NAMES.get(dnp3_func, "UNKNOWN")

            # ── Control byte bit decomposition ──
            dnp3_dir       = (dnp3_ctrl & 0x80) >> 7  # DIR: direction bit
            dnp3_prm       = (dnp3_ctrl & 0x40) >> 6  # PRM: primary message
            dnp3_fcb       = (dnp3_ctrl & 0x20) >> 5  # FCB: frame count bit
            dnp3_fcv       = (dnp3_ctrl & 0x10) >> 4  # FCV: frame count valid
            dnp3_func_code = dnp3_ctrl & 0x0F          # link layer function code

            row = {
                # === File / Frame ===
                "source_file"          : os.path.basename(pcap_path).split(".")[0],
                "frame.number"         : pkt_counter,
                "frame.time_epoch"     : frame_time,
                "frame.time_relative"  : frame_time_relative,
                "frame.time_delta"     : frame_time_delta,
                "frame.len"            : len(pkt),
                "frame.cap_len"        : len(bytes(pkt)),

                # === Ethernet ===
                "eth.src"              : eth_layer.src if eth_layer else None,
                "eth.dst"              : eth_layer.dst if eth_layer else None,

                # === IP ===
                "ip.src"               : ip_layer.src,
                "ip.dst"               : ip_layer.dst,
                "ip.proto"             : ip_layer.proto,
                "ip.ttl"               : ip_layer.ttl,

                # === TCP ===
                "tcp.srcport"          : tcp_layer.sport,
                "tcp.dstport"          : tcp_layer.dport,
                "tcp.flags"            : tcp_layer.flags.value,
                "tcp.len"              : len(tcp_layer.payload),
                "tcp.window_size_value": tcp_layer.window,
                "tcp.checksum"         : tcp_layer.chksum,
                "tcp.time_delta"       : frame_time_delta,

                # === DNP3 Data Link Layer ===
                "dnp3_present"         : True,
                "dnp3.start"           : f"0x{raw[0]:02X}{raw[1]:02X}",
                "dnp3.len"             : dnp3_len,
                "dnp3.ctrl"            : dnp3_ctrl,
                "dnp3.dst_addr"        : dnp3_dst_addr,
                "dnp3.src_addr"        : dnp3_src_addr,
                "dnp3.dir"             : dnp3_dir,
                "dnp3.prm"             : dnp3_prm,
                "dnp3.fcb"             : dnp3_fcb,
                "dnp3.fcv"             : dnp3_fcv,
                "dnp3.func_code_link"  : dnp3_func_code,

                # === DNP3 Application Layer ===
                "dnp3.func_code"       : dnp3_func,
                "dnp3.func_name"       : dnp3_func_name,
                "dnp3.payload_len"     : len(raw),
                "dnp3.raw_hex"         : raw.hex()[:64],

                # === Label (ground truth from folder name) ===
                "label"                : label,
            }
            dnp3_rows.append(row)

    return pd.DataFrame(dnp3_rows)


In [ ]:
def prompt_user() -> tuple:
    """Prompt user for folder path, label, and output path."""

    print("=" * 55)
    print("   DNP3 PCAP Feature Extractor — General Utility")
    print("=" * 55)

    # ── Folder path ──
    while True:
        folder_path = input("\n📂 Enter PCAP folder path: ").strip()
        if os.path.isdir(folder_path):
            break
        print(f"  ❌ Folder not found: {folder_path}")
        print(f"     Please enter a valid folder path.")

    # ── Label ──
    while True:
        user_label = input("\n🏷️  Enter label for ALL packets (e.g. 1, 2, NORMAL, ATTACK): ").strip()
        if user_label:
            break
        print("  ❌ Label cannot be empty. Please enter a value.")

    # ── Output path ──
    default_out = os.path.join(folder_path, "dnp3_extracted.csv")
    out_input   = input(f"\n💾 Enter output CSV path (press Enter for default):\n   [{default_out}]: ").strip()
    output_path = out_input if out_input else default_out

    return folder_path, user_label, output_path

In [ ]:
# ── Main ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # ── Step 1: Get user inputs ──
    folder_path, user_label, output_path = prompt_user()

    # ── Step 2: Find all PCAP files ──
    print(f"\n🔍 Scanning for PCAP files in: {folder_path}")
    pcap_files = get_pcap_files(folder_path)

    if len(pcap_files) == 0:
        print("❌ No PCAP files found. Check the folder path.")
        exit()

    print(f"✅ Found {len(pcap_files)} PCAP files")
    for f in pcap_files:
        print(f"   • {os.path.basename(f)}")

    # ── Step 3: Confirm ──
    print(f"\n{'─'*55}")
    print(f"  Folder  : {folder_path}")
    print(f"  Label   : {user_label}  (applied to ALL packets)")
    print(f"  Output  : {output_path}")
    print(f"{'─'*55}")
    confirm = input("\n▶  Press Enter to start extraction (or 'q' to quit): ").strip()
    if confirm.lower() == 'q':
        print("Cancelled.")
        exit()

    # ── Step 4: Extract features ──
    print()
    df = parse_pcap_files(pcap_files, user_label)

    # ── Step 5: Results ──
    if len(df) == 0:
        print("\n⚠️  No DNP3 packets found.")
        print("   Check if PCAP files contain DNP3 traffic on ports 20000-20003")
        exit()

    print(f"\n{'='*55}")
    print(f"  ✅ Extraction Complete!")
    print(f"{'='*55}")
    print(f"  Total packets : {len(df)}")
    print(f"  Label applied : {user_label}")
    print(f"  Shape         : {df.shape}")
    print(f"  Columns       : {len(df.columns)}")

    # ── Step 6: Save CSV ──
    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"\n💾 Saved to: {output_path}")

    # ── Step 7: Preview ──
    print(f"\n📊 Label distribution:")
    print(df['label'].value_counts().to_string())
    print(f"\n📋 First 3 rows:")
    print(df[['source_file', 'ip.src', 'ip.dst', 'dnp3.func_name', 'label']].head(3).to_string())

   DNP3 PCAP Feature Extractor — General Utility

📂 Enter PCAP folder path: /content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Fina
  ❌ Folder not found: /content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Fina
     Please enter a valid folder path.

📂 Enter PCAP folder path: /content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final

🏷️  Enter label for ALL packets (e.g. 1, 2, NORMAL, ATTACK): 1

💾 Enter output CSV path (press Enter for default):
   [/content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/dnp3_extracted.csv]: /content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final/dnp3_extracted_1.csv

🔍 Scanning for PCAP files in: /content/drive/MyDrive/ DNP3 dataset/DNP3_Intrusion_Detection_Dataset_Final
✅ Found 156 PCAP files
   • 20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_01.pcap
   • 20200515_DNP3_Cold_Restart_Attack_UOWM_DNP3_Dataset_Slave_02.pcap
   • 20200515_DNP3_Cold_

Processing PCAP files:   0%|          | 0/156 [00:00<?, ?file/s]